# Домашнее задание 3. Парсинг, Git и тестирование на Python

**Цели задания:**

* Освоить базовые подходы к web-scraping с библиотеками `requests` и `BeautisulSoup`: навигация по страницам, извлечение HTML-элементов, парсинг.
* Научиться автоматизировать задачи с использованием библиотеки `schedule`.
* Попрактиковаться в использовании Git и оформлении проектов на GitHub.
* Написать и запустить простые юнит-тесты с использованием `pytest`.


В этом домашнем задании вы разработаете систему для автоматического сбора данных о книгах с сайта [Books to Scrape](http://books.toscrape.com). Нужно реализовать функции для парсинга всех страниц сайта, извлечения информации о книгах, автоматического ежедневного запуска задачи и сохранения результата.

Важной частью задания станет оформление проекта: вы создадите репозиторий на GitHub, оформите `README.md`, добавите артефакты (код, данные, отчеты) и напишете базовые тесты на `pytest`.



In [1]:
! pixi add schedule pytest beautifulsoup4 
# ! pip install -q schedule pytest  # установка библиотек, если ещё не

✔ schedule is already a dependency11m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━]    0/0                                                                             
  Run `pixi upgrade schedule` to get the newest compatible version
✔ pytest is already a dependency
  Run `pixi upgrade pytest` to get the newest compatible version
✔ beautifulsoup4 is already a dependency
  Run `pixi upgrade beautifulsoup4` to get the newest compatible version


In [2]:
# Библиотеки, которые могут вам понадобиться
# При необходимости расширяйте список
import time
import requests
import schedule
from bs4 import BeautifulSoup

## Задание 1. Сбор данных об одной книге (2 баллов)

В этом задании мы начнем подготовку скрипта для парсинга информации о книгах со страниц каталога сайта [Books to Scrape](https://books.toscrape.com/).

Для начала реализуйте функцию `get_book_data`, которая будет получать данные о книге с одной страницы (например, с [этой](http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)). Соберите всю информацию, включая название, цену, рейтинг, количество в наличии, описание и дополнительные характеристики из таблицы Product Information. Результат достаточно вернуть в виде словаря.

**Не забывайте про соблюдение PEP-8** — помимо качественно написанного кода важно также документировать функции по стандарту:
* кратко описать, что она делает и для чего нужна;
* какие входные аргументы принимает, какого они типа и что означают по смыслу;
* аналогично описать возвращаемые значения.

*P. S. Состав, количество аргументов функции и тип возвращаемого значения можете менять как вам удобно. То, что написано ниже в шаблоне — лишь пример.*

In [16]:
def get_book_data(book_url: str) -> dict:
    """Возвращает данные книги и характеристики из Product Information.

    Args:
        book_url: URL страницы книги с таблицей характеристик.

    Returns:
        Словарь с title (название), rating (целое число от 1 до 5),
        description (описание или пустая строка, если его нет) и строковыми
        характеристиками из таблицы. Пробелы по краям текста удалены.

    Raises:
        requests.RequestException: Не удалось загрузить страницу книги.
    """
    # НАЧАЛО ВАШЕГО РЕШЕНИЯ
    resp = requests.get(book_url)
    soup = BeautifulSoup(resp.content)

    # На странице с описанием книги только один тег таблица
    table = soup.find("table")

    data = {}
    for row in table.find_all("tr"):
        key = row.find("th").get_text(strip=True)
        value = row.find("td").get_text(strip=True)
        data[key] = value

    data["title"] = soup.find("h1").get_text(strip=True)

    ratings = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
    rating_classes = soup.find("p", class_="star-rating")["class"]
    data["rating"] = next(
        ratings[name] for name in rating_classes if name in ratings
    )

    description = soup.select_one("#product_description + p")
    data["description"] = (
        description.get_text(" ", strip=True) if description else ""
    )

    return data
    # КОНЕЦ ВАШЕГО РЕШЕНИЯ


In [17]:
# Используйте для самопроверки
book_url = 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'
get_book_data(book_url)

{'UPC': 'a897fe39b1053632',
 'Product Type': 'Books',
 'Price (excl. tax)': '£51.77',
 'Price (incl. tax)': '£51.77',
 'Tax': '£0.00',
 'Availability': 'In stock (22 available)',
 'Number of reviews': '0',
 'title': 'A Light in the Attic',
 'rating': 3,
 'description': "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh

## Задание 2. Сбор данных обо всех книгах (2 баллов)

Создайте функцию `scrape_books`, которая будет проходиться по всем страницам из каталога (вида `http://books.toscrape.com/catalogue/page-{N}.html`) и осуществлять парсинг всех страниц в цикле, используя ранее написанную `get_book_data`.

Добавьте аргумент-флаг, который будет отвечать за сохранение результата в файл: если он будет равен `True`, то информация сохранится в ту же папку в файл `books_data.txt`; иначе шаг сохранения будет пропущен.

**Также не забывайте про соблюдение PEP-8**

In [18]:
import json
from pathlib import Path
from urllib.parse import urljoin


def save_file(data: str, filepath: Path):
    """Записывает текст в файл, заменяя его содержимое, если он существует.

    Args:
        data: Текст для записи.
        filepath: Путь к файлу в существующем каталоге.

    Raises:
        OSError: Не удалось записать файл.
    """
    filepath.write_text(data)


def scrape_books(
    is_save: bool,
    left: int,
    right: int,
    catalogue: str | None = None,
    filepath: Path | None = None,
):
    """Собирает характеристики книг с необязательным сохранением в JSON.

    Args:
        is_save: Сохранить результат в файл в формате JSON.
        left: Номер первой страницы для сбора, начиная с 1.
        right: Верхняя граница номеров страниц, не включая её.
        catalogue: Путь относительно /catalogue/category/, например
            "books_1". Если не задан, используются ссылки бокового каталога.
        filepath: Путь к файлу результата. По умолчанию — books_data.txt
            в текущем рабочем каталоге. Учитывается только при is_save=True;
            существующий файл перезаписывается.

    Returns:
        Словарь, где ключи — URL каталогов, а значения — списки словарей
        характеристик книг, возвращённых get_book_data.

    Raises:
        requests.RequestException: Не удалось загрузить страницу.
        OSError: Не удалось сохранить результат при is_save=True.
    """
    # НАЧАЛО ВАШЕГО РЕШЕНИЯ
    base_url = "http://books.toscrape.com"
    resp = requests.get(f"{base_url}/index.html")
    index_page = BeautifulSoup(resp.content)

    if not catalogue:
        catalogues_links = [
            link.get("href")
            for link in index_page.find("div", class_="side_categories").find_all("a")
        ]
        catalogues = [urljoin(base_url, link) for link in catalogues_links]
    else:
        catalogues = [urljoin(base_url, f"/catalogue/category/{catalogue}/")]

    catalogue_books_map = {}
    for cat in catalogues:
        books_data = []
        for n in range(left, right):
            page_url = urljoin(cat, f"page-{n}.html")

            resp = requests.get(page_url)
            if resp.status_code == 404 and n != 1:
                break

            if resp.status_code == 404:
                page_url = urljoin(cat, "index.html")
                resp = requests.get(page_url)

            category_page = BeautifulSoup(resp.content)

            books_links = category_page.find("section").find_all("a")
            navigation_links = category_page.select("li.previous a, li.next a")

            hrefs = [
                link.get("href").split("../")[-1]
                for link in books_links
                if link not in navigation_links
            ]

            books = {urljoin(base_url, f"/catalogue/{ref}") for ref in hrefs}

            for book in books:
                books_data.append(get_book_data(book))

        catalogue_books_map[cat] = books_data

    if is_save:
        filepath = filepath or Path("books_data.txt")
        save_file(json.dumps(catalogue_books_map, ensure_ascii=False), filepath)
    return catalogue_books_map

    # КОНЕЦ ВАШЕГО РЕШЕНИЯ


In [19]:
# Проверка работоспособности функции
res = scrape_books(is_save=True, left=1, right=2, catalogue="books_1") # Допишите ваши аргументы
print(type(res), len(res)) # и проверки

<class 'dict'> 1


## Задание 3. Настройка регулярной выгрузки (1 баллов)

Настройте автоматический запуск функции сбора данных каждый день в 19:00.
Для автоматизации используйте библиотеку `schedule`. Функция должна запускаться в указанное время и сохранять обновленные данные в текстовый файл.



Бесконечный цикл должен обеспечивать постоянное ожидание времени для запуска задачи и выполнять ее по расписанию. Однако чтобы не перегружать систему, стоит подумать о том, чтобы выполнять проверку нужного времени не постоянно, а раз в какой-то промежуток. В этом вам может помочь `time.sleep(...)`.

Проверьте работоспособность кода локально на любом времени чч:мм.



In [7]:
# НАЧАЛО ВАШЕГО РЕШЕНИЯ

schedule.every().day.at("19:00").do(scrape_books, is_save=True, left=1, right=2, catalogue="books_1")

while True:
    schedule.run_pending()
    # 3600 секунд спать, чтобы не пропустить запуск и чтобы не загружать процессор в busy sleep.
    # В самом плохом случае пропустим на час.
    time.sleep(60*60)

# КОНЕЦ ВАШЕГО РЕШЕНИЯ

KeyboardInterrupt: 

## Задание 4. Написание автотестов (1,5 баллов)

Создайте минимум три автотеста для ключевых функций парсинга — например, `get_book_data` и `scrape_books`. Идеи проверок (можете использовать свои):

* данные о книге возвращаются в виде словаря с нужными ключами;
* список ссылок или количество собранных книг соответствует ожиданиям;
* значения отдельных полей (например, `title`) корректны.

Оформите тесты в отдельном скрипте `tests/test_scraper.py`, используйте библиотеку `pytest`. Убедитесь, что тесты проходят успешно при запуске из терминала командой `pytest`.

Также выведите результат их выполнения в ячейке ниже.

**Не забывайте про соблюдение PEP-8**


In [14]:
# Ячейка для демонстрации работоспособности
# Сам код напишите в отдельном скрипте
# ! pytest tests/test_scraper.py
! pixi run pytest ../tests/test_scraper.py

⠉ activating environment                                                        ============================= test session starts ==============================
platform darwin -- Python 3.14.10, pytest-9.2.0, pluggy-1.7.0
rootdir: /Users/kirsh-mac/masters/programming_python
plugins: platformdirs-4.12.4, anyio-4.15.1
collected 3 items                                                              

../tests/test_scraper.py ...                                             [100%]

============================== 3 passed in 0.14s ===============================


## Задание 5. Оформление проекта на GitHub и работа с Git (3,5 баллов)

В этом задании нужно воспользоваться системой контроля версий Git и платформой GitHub для хранения и управления своим проектом. **Ссылку на свой репозиторий пришлите в форме для сдачи ответа.**

### Пошаговая инструкция и задания

**1. Установите Git на свой компьютер.**

* Для Windows: [скачайте установщик](https://git-scm.com/downloads) и выполните установку.
* Для macOS:

  ```
  brew install git
  ```
* Для Linux:

  ```
  sudo apt update
  sudo apt install git
  ```

**2. Настройте имя пользователя и email.**

Это нужно для подписи ваших коммитов, сделайте в терминале через `git config ...`.

**3. Создайте аккаунт на GitHub**, если у вас его еще нет:
[https://github.com](https://github.com)

**4. Создайте новый репозиторий на GitHub:**

* Найдите кнопку **New repository**.
* Укажите название, краткое описание, выберите тип **Public** (чтобы мы могли проверить ДЗ).
* Не ставьте галочку Initialize this repository with a README.

**5. Создайте локальную папку с проектом.** Можно в терминале, можно через UI, это не имеет значения.

**6. Инициализируйте Git в этой папке.** Здесь уже придется воспользоваться некоторой командой в терминале.

**7. Привяжите локальный репозиторий к удаленному на GitHub.**

**8. Создайте ветку разработки.** По умолчанию вы будете находиться в ветке `main`, создайте и переключитесь на ветку `hw-books-parser`.

**9. Добавьте в проект следующие файлы и папки:**

* `scraper.py` — ваш основной скрипт для сбора данных.
* `README.md` — файл с кратким описанием проекта:

  * цель;
  * инструкции по запуску;
  * список используемых библиотек.
* `requirements.txt` — файл со списком зависимостей, необходимых для проекта (не присылайте все из глобального окружения, создайте изолированную виртуальную среду, добавьте в нее все нужное для проекта и получите список библиотек через `pip freeze`).
* `artifacts/` — папка с результатами парсинга (`books_data.txt` — полностью или его часть, если весь не поместится на GitHub).
* `notebooks/` — папка с заполненным ноутбуком `HW_03_python_ds.ipynb` и запущенными ячейками с выводами на экран.
* `tests/` — папка с тестами на `pytest`, оформите их в формате скрипта(-ов) с расширением `.py`.
* `.gitignore` — стандартный файл, который позволит исключить временные файлы при добавлении в отслеживаемые (например, `__pycache__/`, `.DS_Store`, `*.pyc`, `venv/` и др.).


**10. Сделайте коммит.**

**11. Отправьте свою ветку на GitHub.**

**12. Создайте Pull Request:**

* Перейдите в репозиторий на GitHub.
* Нажмите кнопку **Compare & pull request**.
* Укажите, что было добавлено, и нажмите **Create pull request**.

**13. Выполните слияние Pull Request:**

* Убедитесь, что нет конфликтов.
* Нажмите **Merge pull request**, затем **Confirm merge**.

**14. Скачайте изменения из основной ветки локально.**



### Требования к итоговому репозиторию

* Файл `scraper.py` с рабочим кодом парсера.
* `README.md` с описанием проекта и инструкцией по запуску.
* Папка `artifacts/` с результатом сбора данных (`.txt` файл).
* Папка `tests/` с тестами на `pytest`.
* Папка `notebooks/` с заполненным ноутбуком `HW_03_python_ds_2025.ipynb`.
* Pull Request с комментарием из ветки `hw-books-parser` в ветку `main`.
* Примерная структура:

  ```
  books_scraper/
  ├── artifacts/
  │   └── books_data.txt
  ├── notebooks/
  │   └── HW_03_python_ds_2025.ipynb
  ├── scraper.py
  ├── README.md
  ├── tests/
  │   └── test_scraper.py
  ├── .gitignore
  └── requirements.txt
  ```

## Выводы, рефлексия и сдача

### Рефлексия после выполнения (заполните здесь)

**1. Что было задано и что получилось**:

`Ответ: Было задано создать веб-скрапер учебного сайта с книгами, все получилось сделать. У меня получилось выполнить задания 1 и 3, но пришлось спросить у Codex(промпты ниже), как правильно работать с bs4, потому что документация бывает запутанная и не совсем были понятны некоторые тонкости работы. Над 2 задачей было интересно работать и она получилась путем проб и ошибок самостоятельно, не считая промпта Codex для прояснения деталей bs4. Задания 4 и 5 оказались очень приятными, потому что я использовал DI, чтобы было легче тестировать функционал - без мока запроса requests.`

**2. Как выполняли**: что сработало сразу, где застряли и как выбрались:

`Ответ: Выполнял задания по-порядку застрял на 1-й и на 3-й задачах, как описал выше воспользовался LLM для помощи с bs4.`

**3. Использование ИИ**: цель, использованные промпты, где помощник ошибся и как вы это поняли. Если не использовали, так и напишите:

`Ответ: Я использовал промпт в конце завершения работы: "Проверь правильность работы и подсвети слабые места работы." Я также использовал промпты для помощи себе в понимание bs4 при поиске тегов в странице: "Как мне достать таблицу из html при помощи beautifulsoup4 и сформировать dict в python.". При проверке LLM-кой были выявлены слабые места в тестах, где не хватало теста на метод scrape_books. Я попросил добавить этот тест и провалидировал его. Я также попросил LLM-ку описать примеры использования Scraper в README.md и отредактировать комментарии методов и функций, чтобы они были понятным и емкими.`

**4. Что осталось непонятным** (честный список, он поможет на консультации):

`Ответ: Интерфейсы bs4 и urllib все еще кажутся запутанными, но это врядли стоит объяснять отдельно. Все остальное ясно.`

**5. Самооценка по критериям задания**:

`Ответ: Я оцениваю себя на 9/10, так как я написал не самый оптимальный код и тесты нужно на большее количество краевых случаев.`

**6. Дополнительный вопрос: обратная связь по заданию.** Что было ценно, что сложно, а что, на ваш взгляд, требует доработки?

`Ответ: В задании все хорошо, мне понравилось.`

> Напоминание: рефлексия пишется своими словами, без помощи ИИ.

